# 01 · Ingesta, normalización y segmentación

Procesar los originales de HTML y PDF y conservar la procedencia de cada unidad. La implementación está en `scripts/auxiliares/ingesta.py`. Los originales no se modifican.

## Entorno

El mismo notebook corre en local o en Colab. En Colab, subir ai-week-colab.zip a Mi unidad/AIWEEK y seleccionar GPU A100. El ZIP ya incluye los datos necesarios.

Los datos se extraen al disco de la sesión. Los resultados nuevos se guardan en Drive mientras avanza la ejecución. Los modelos se descargan al disco de Colab. Si se elimina la sesión, se descargan de nuevo.

Si la instalación pide reiniciar, reiniciar la sesión y volver a ejecutar desde esta celda. MODO puede ser auto, local o colab.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import shutil
import subprocess
import sys
import zipfile

MODO = "auto"
CARPETA_DRIVE = "/content/drive/MyDrive/AIWEEK"
try:
    import google.colab
    detectado_colab = True
except ImportError:
    detectado_colab = False
if MODO not in {"auto", "local", "colab"}:
    raise ValueError("MODO debe ser auto, local o colab")
EN_COLAB = detectado_colab if MODO == "auto" else MODO == "colab"

def hash_archivo(ruta):
    h = hashlib.sha256()
    with Path(ruta).open("rb") as archivo:
        for bloque in iter(lambda: archivo.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(CARPETA_DRIVE)
    paquete_drive = DRIVE / "ai-week-colab.zip"
    if not paquete_drive.is_file():
        raise FileNotFoundError(f"Subir ai-week-colab.zip a {DRIVE}")
    paquete = Path("/content/ai-week-colab.zip")
    shutil.copy2(paquete_drive, paquete)
    huella_paquete = hash_archivo(paquete)
    raiz = Path("/content/aiweek") / huella_paquete[:12]
    if not (raiz / ".paquete_verificado").is_file():
        raiz.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(paquete) as z:
            for nombre in z.namelist():
                if not (raiz / nombre).resolve().is_relative_to(raiz.resolve()):
                    raise ValueError("Ruta inválida dentro del ZIP")
            z.extractall(raiz)
        inventario = json.loads((raiz / "paquete.json").read_text(encoding="utf-8"))
        for nombre, esperado in inventario["sha256_archivos"].items():
            if hash_archivo(raiz / nombre) != esperado:
                raise ValueError(f"Archivo incompleto: {nombre}")
        (raiz / ".paquete_verificado").write_text(huella_paquete)
    dependencias = raiz / "requirements-colab.txt"
    marca = Path("/content/aiweek_dependencias.sha256")
    version_deps = hash_archivo(dependencias)
    if not marca.exists() or marca.read_text() != version_deps:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(dependencias)], check=True)
        marca.write_text(version_deps)
    for nombre in ("numpy", "pandas", "transformers", "tokenizers"):
        if nombre in sys.modules and getattr(sys.modules[nombre], "__version__", None) != importlib.metadata.version(nombre):
            raise RuntimeError("Reiniciar la sesión de Colab y ejecutar desde el principio para cargar las versiones instaladas")
    RESULTADOS = DRIVE / "resultados" / huella_paquete[:12]
else:
    raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/modelos.json").is_file()), None)
    if raiz is None:
        raise FileNotFoundError("Abrir el notebook desde la carpeta del proyecto o notebooks")
    RESULTADOS = raiz / "data/experimentos_oficiales"
RAIZ = raiz
RESULTADOS.mkdir(parents=True, exist_ok=True)
if EN_COLAB:
    corpus_drive = RESULTADOS / "corpus_procesado.zip"
    marca_corpus = raiz / ".corpus_restaurado"
    if corpus_drive.is_file():
        huella_corpus = hash_archivo(corpus_drive)
        if not marca_corpus.exists() or marca_corpus.read_text() != huella_corpus:
            carpeta_corpus = raiz / "data/processed/corpus"
            with zipfile.ZipFile(corpus_drive) as z:
                for nombre in z.namelist():
                    if not (carpeta_corpus / nombre).resolve().is_relative_to(carpeta_corpus.resolve()):
                        raise ValueError("Ruta inválida en el corpus guardado")
                z.extractall(carpeta_corpus)
            marca_corpus.write_text(huella_corpus)
os.chdir(raiz)
sys.path.insert(0, str(raiz))
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["HF_HOME"] = str(raiz / "data/cache_huggingface") if EN_COLAB else os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
print("Entorno:", "Colab" if EN_COLAB else "local")
print("Proyecto:", raiz)
print("Resultados:", RESULTADOS)

## 1. Selección

El modo corpus procesa los 282 documentos. El modo muestra permite revisar los identificadores elegidos antes de exportar. La extracción verifica cada original contra el manifiesto.

In [1]:
from pathlib import Path
import sys
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / "data/raw/manifest.json").exists():
    RAIZ = RAIZ.parent
if not (RAIZ / "data/raw/manifest.json").exists():
    raise FileNotFoundError("Abrir el notebook desde la carpeta del proyecto o notebooks")
sys.path.insert(0, str(RAIZ))
from scripts.auxiliares.ingesta import ejecutar_ingesta, guardar_resultados

MODO = "corpus"
IDS_MUESTRA = ["co_decreto_306_1992", "ley_1032_2006"]
SALIDA = RAIZ / "data/processed" / MODO
MAX_CARACTERES = 1800
SOLAPAMIENTO = 200

## 2. Extracción y segmentación

Se normalizan espacios y codificación. Se conservan cifras, referencias, notas y tachados explícitos. Los PDF conservan las páginas y quedan señalados para revisar su orden de lectura. Las cabeceras repetidas de EVA retiradas quedan registradas. No se realiza OCR.

Los artículos de las normas forman unidades completas. Los encabezados partidos o pegados se reconocen sin cambiar el texto. Las citas y reformas permanecen dentro del artículo que las introduce. Las atribuciones ambiguas quedan nulas. Las versiones anteriores explícitas se conservan con su marca.

La jurisprudencia mantiene el artículo nulo. Se separan secciones y numerales, y los tramos extensos se dividen en bloques técnicos de hasta 6000 caracteres. Los índices internos se conservan fuera de la búsqueda. Las ventanas de hasta 1800 caracteres solo localizan su unidad, con solapamiento de hasta 200 dentro de ella.

In [2]:
resultado = ejecutar_ingesta(
    RAIZ, modo=MODO, ids_muestra=IDS_MUESTRA,
    max_chars=MAX_CARACTERES, solapamiento=SOLAPAMIENTO,
)
registros = resultado["registros"]
unidades = resultado["unidades"]
fragmentos = resultado["fragmentos"]
resumen = resultado["resumen"]
pd.Series(resumen).to_frame("resultado")

Procesados: 25/282


Procesados: 50/282


Procesados: 75/282


Procesados: 100/282


Procesados: 125/282


...

Procesados: 225/282


Procesados: 250/282


Procesados: 275/282


Procesados: 282/282


,resultado
version_ingesta,0.3.0
modo,corpus
sha256_manifest,7820b44893726d7a65a77e7e0dc73d8e8e63954a3802a4...
sha256_parser,86822dd939b093072db17ffe6fe466d9f24f394b356aa2...
documentos,282
textos_guardados,282
estados_extraccion,"{'extraido': 220, 'revisar': 62}"
unidades,55449
unidades_por_tipo,"{'bloque_jurisprudencia': 2479, 'indice': 2, '..."
fragmentos,71402


## 3. Comprobaciones

Se comprueban cobertura, offsets, procedencia, límites y ejemplos revisados contra el original. Las discontinuidades son señales para revisión. No demuestran que falten artículos ni certifican la vigencia.

In [3]:
comprobaciones = pd.DataFrame(resultado["comprobaciones"])
display(comprobaciones)
if not comprobaciones["resultado"].eq("ok").all():
    raise ValueError("Hay comprobaciones fallidas. No exportar los derivados.")

,comprobacion,resultado
0,Extracción de todos los documentos,ok
1,Todos los documentos tienen fragmentos,ok
2,Identificadores únicos,ok
3,Substrings exactos y límites de ventana,ok
4,Ventanas dentro de su unidad,ok
5,Unidades exactas,ok
6,Unidades sin solapamiento,ok
7,Sin artículos propios en jurisprudencia,ok
8,Bloques judiciales acotados,ok
9,Procedencia de todos los fragmentos,ok


In [4]:
continuidad = pd.DataFrame(resultado["continuidad"])
display(continuidad.loc[continuidad["encabezados_ambiguos"].gt(0),
    ["doc_id", "candidatos", "unidades_con_articulo", "encabezados_ambiguos", "saltos_simples_por_revisar"]].head(20))
pd.Series(resumen["avisos_unidades"], name="unidades").sort_values(ascending=False).to_frame()

,doc_id,candidatos,unidades_con_articulo,encabezados_ambiguos,saltos_simples_por_revisar
1,co_constitucion_1_1991,1513,405,1108,"[[1, 7], [7, 14], [19, 28], [28, 33], [40, 44]..."
2,co_decreto_1069_2015,1490,1234,255,[]
3,co_decreto_1082_2015,1068,840,228,[]
4,co_decreto_1083_2015,1002,904,95,[]
11,co_ley_136_1994,205,203,2,[]
12,co_ley_137_1994,8,7,1,"[[4, 7]]"
14,co_ley_1551_2012,58,26,16,"[[4, 7], [12, 15], [16, 19], [19, 22], [22, 25..."
18,co_ley_1755_2015,23,2,20,[]
19,co_ley_1757_2015,113,111,2,[]
20,co_ley_1952_2019,280,279,1,"[[77, 80]]"


,unidades
atribucion_articulo_ambigua,5472
anotaciones_editoriales,3723
revisar_anotaciones_y_publicacion,3723
revisar_orden_y_texto_pdf,2230
incluye_articulo_citado_o_reformado,1797
posible_mojibake_revisar_fuente,1217
limite_siguiente_con_atribucion_ambigua,707
original_pdf_sin_cierre_completo,301
articulo_citado_sin_unidad_reformadora,97
version_anterior_conservada,11


## 4. Fuentes pendientes

SC5191, C264 y Ley 1551 tienen defectos confirmados en la auditoría del original. Se conservan, pero quedan fuera de búsqueda mientras se revisan. La restricción está ligada al hash auditado. Tener texto extraído no garantiza una fuente completa ni una norma vigente.

In [5]:
pd.DataFrame([{
    "doc_id": r["doc_id"], "estado_fuente": r.get("estado_fuente"),
    "motivos": [m["motivo"] for m in r.get("rechazos_fuente", [])]
} for r in registros if r.get("apta_para_busqueda") is False])

,doc_id,estado_fuente,motivos
0,co_ley_1551_2012,defecto_confirmado,[mojibake_irrecuperable_confirmado_Ley1551]
1,sentencia_cc_c264_2026,defecto_confirmado,[mojibake_residual_confirmado_C264]
2,sentencia_csj_sc5191_2020,defecto_confirmado,[OCR_erroneo_confirmado_SC5191]


## 5. Exportación

Se guardan textos, documentos, unidades completas, ventanas, continuidad, resumen y comprobaciones en `data/processed/corpus` o `data/processed/muestra`. Los offsets se refieren al texto normalizado. El resumen conserva los hashes del manifiesto y del parser.

La repetición actualiza estos archivos. Los TXT antiguos permanecen y `documentos.jsonl` identifica la ejecución actual.

In [6]:
guardar_resultados(SALIDA, resultado)
print("Guardado en:", SALIDA)
print("Documentos:", resumen["documentos"])
print("Unidades:", resumen["unidades"])
print("Ventanas:", resumen["fragmentos"])

Guardado en: C:\Users\nrir\Desktop\dev\ai-week-2026\data\processed\corpus
Documentos: 282
Unidades: 55449
Ventanas: 71402


## Resultados

Se procesaron 282 documentos y 1304 páginas PDF. Se guardaron 55449 unidades y 71402 ventanas de hasta 1800 caracteres. Pasaron las 22 comprobaciones.

Los cuatro controles normativos conservaron 10, 5, 30 y 45 artículos. No quedan los tres microcortes identificados en el EDA. Los bloques judiciales tienen hasta 6000 caracteres.

Persisten 5569 encabezados ambiguos en 95 documentos, 11 versiones anteriores y 62 documentos con avisos de extracción. Hay 14 PDF con cierre incompleto y 44 señales de codificación por revisar. Tres fuentes quedan fuera de búsqueda.

## Decisiones

- Recuperar artículos completos como evidencia y usar las ventanas para localizarlos.
- Mantener nulas las atribuciones ambiguas y las de jurisprudencia.
- Revisar los avisos y las versiones anteriores antes de indexar.
- Conservar las marcas de vigencia sin afirmar que la norma sigue vigente.


## Guardado en Drive

En Colab se conserva el corpus procesado para los siguientes notebooks y sesiones.

In [ ]:
if EN_COLAB and MODO == "corpus":
    archivo = shutil.make_archive(str(raiz / "data/corpus_procesado"), "zip", SALIDA)
    shutil.copy2(archivo, RESULTADOS / "corpus_procesado.zip")
    print("Corpus guardado en", RESULTADOS / "corpus_procesado.zip")
